<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/Decision__Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
newdf = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

In [17]:
newdf['product_name'].value_counts().value_counts()

,count
count,
1,1168
2,18
3,4
4,2


In [3]:
newdf.head()

,skintype,skin_condition,product_type,product_name,brand,notable_effects,picture_src
0,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Face Wash,ACWELL Bubble Free PH Balancing Cleanser,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
1,"Normal, Dry, Combination","('redness', 'skin imbalance')",Face Wash,ACWELL pH Balancing Soothing Cleansing Foam,ACWELL,"('soothing', 'balancing')",https://images.soco.id/8f08ced0-344d-41f4-a15e...
2,"Normal, Dry, Oily, Combination, Sensitive","('redness', 'skin imbalance')",Toner,Acwell Licorice pH Balancing Cleansing Toner,ACWELL,"('soothing', 'balancing')","https://www.soco.id/cdn-cgi/image/w=73,format=..."
3,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Toner,ACWELL Aquaseal Soothing Tonic,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
4,"Normal, Dry","('pigmentation', 'redness')",Toner,Licorice pH Balancing Essence Mist,ACWELL,"('brightening', 'soothing')","https://www.sociolla.com/cdn-cgi/image/w=425,f..."


In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report

In [6]:
data = newdf[['skintype','skin_condition','product_type','product_name']]

In [7]:
data['skintype'] = data['skintype'].apply(lambda x: [i.strip() for i in str(x).split(',')])

data['skin_condition'] = data['skin_condition'].apply(
    lambda x: str(x).replace("(","").replace(")","").replace("'","").split(',')
)

/tmp/ipykernel_768/1698155163.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['skintype'] = data['skintype'].apply(lambda x: [i.strip() for i in str(x).split(',')])
/tmp/ipykernel_768/1698155163.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['skin_condition'] = data['skin_condition'].apply(


In [9]:
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
mlb_skin = MultiLabelBinarizer()
skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

mlb_condition = MultiLabelBinarizer()
condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

In [10]:
le_type = LabelEncoder()

data['product_type'] = le_type.fit_transform(data['product_type'])

/tmp/ipykernel_768/4224783904.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['product_type'] = le_type.fit_transform(data['product_type'])


In [18]:
le_product_type = LabelEncoder()

y = le_product_type.fit_transform(data['product_type'])

In [38]:
effects_features = MultiLabelBinarizer().fit_transform(newdf['notable_effects'])

In [47]:
X = pd.concat([
    skin_features,
    condition_features,
    pd.DataFrame(effects_features, columns=MultiLabelBinarizer().fit(newdf['notable_effects']).classes_)
], axis=1)

In [86]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [92]:
print("Train Accuracy:", model.score(X_train, y_train))
print("Test Accuracy:", model.score(X_test, y_test))

Train Accuracy: 0.6731358529111338
Test Accuracy: 0.5183673469387755


In [91]:
importance = pd.Series(model.feature_importances_, index=X.columns)

low_features = importance[importance < 0.01].index
X = X.drop(columns=low_features)

In [97]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=25,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=25, min_samples_leaf=2, min_samples_split=4,
                       random_state=42)

In [98]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [10, 15, 20, 25],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt", "log2"]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
model = grid.best_estimator_

Best Parameters: {'criterion': 'gini', 'max_depth': 15, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 10}


In [99]:
model = grid.best_estimator_

In [100]:
y_pred = model.predict(X_test)

In [101]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.5306122448979592
              precision    recall  f1-score   support

           0       0.51      0.60      0.55        40
           1       0.27      0.38      0.32        39
           2       0.57      0.66      0.61        64
           3       0.77      0.71      0.73        51
           4       0.59      0.25      0.36        51

    accuracy                           0.53       245
   macro avg       0.54      0.52      0.51       245
weighted avg       0.56      0.53      0.53       245



In [57]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5183673469387755


In [61]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(100))

UV-Protection    0.266034
Sensitive        0.074313
Combination      0.072361
Normal           0.071408
Brightening      0.069904
Dry              0.061910
Acne-Free        0.055931
Pore-Care        0.050657
Oily             0.043251
Anti-Aging       0.042581
Moisturizing     0.041948
Hydrating        0.029154
Soothing         0.027871
Balancing        0.023397
Refreshing       0.020141
Oil-Control      0.018863
Skin-Barrier     0.015612
Black-Spot       0.011313
No-Whitecast     0.003350
Acne-Spot        0.000000
Oil-control      0.000000
dtype: float64


In [109]:
# ==============================
# USER INPUT
# ==============================

print("Select Skin Type:\n")
skin_list = list(mlb_skin.classes_)

for i, s in enumerate(skin_list):
    print(i, ":", s)

skin_choice = int(input("\nEnter number for Skin Type: "))


print("\nSelect Skin Concern / Effect:\n")
effect_list = list(mlb_effects.classes_)

for i, e in enumerate(effect_list):
    print(i, ":", e)

effect_choice = int(input("\nEnter number for Effect: "))


# Convert selection to text
selected_skin = skin_list[skin_choice]
selected_effect = effect_list[effect_choice]


# ==============================
# BUILD INPUT WITH SAME FEATURES
# ==============================

import pandas as pd

# Create empty dataframe with same columns as training data
user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

# Activate selected features
if selected_skin in user_df.columns:
    user_df.loc[0, selected_skin] = 1

if selected_effect in user_df.columns:
    user_df.loc[0, selected_effect] = 1


# ==============================
# MODEL PREDICTION
# ==============================

prediction = model.predict(user_df)

predicted_type = le_product_type.inverse_transform(prediction)

print("\nRecommended Product Type:", predicted_type[0])


# ==============================
# SHOW RECOMMENDED PRODUCTS
# ==============================

recommended_products = df[df["product_type"] == predicted_type[0]].head(5)

print("\nTop Recommended Products:\n")

for _, row in recommended_products.iterrows():
    print("Product:", row["product_name"])
    print("Brand:", row["brand"])
    print("Image:", row["picture_src"])
    print("-"*40)

Select Skin Type:

0 : Combination
1 : Dry
2 : Normal
3 : Oily
4 : Sensitive

Enter number for Skin Type: 4

Select Skin Concern / Effect:

0 : Acne-Free
1 : Acne-Spot
2 : Anti-Aging
3 : Balancing
4 : Black-Spot
5 : Brightening
6 : Hydrating
7 : Moisturizing
8 : No-Whitecast
9 : Oil-Control
10 : Oil-control
11 : Pore-Care
12 : Refreshing
13 : Skin-Barrier
14 : Soothing
15 : UV-Protection

Enter number for Effect: 2

Recommended Product Type: Serum

Top Recommended Products:

Product: Licorice pH Balancing Advance Serum
Brand: ACWELL 
Image: https://www.sociolla.com/cdn-cgi/image/w=425,format=auto,dpr=1.45/https://images.soco.id/b6bbd6a7-56d5-4b89-9e95-d69149c87f7f-image-0-1610606096802
----------------------------------------
Product: AHC Peony Bright Luminous Serum
Brand: AHC
Image: https://www.beautyhaul.com/assets/uploads/products/thumbs/800x800/PEONY_BRIGHT_LUMINOUS_SERUM.jpg
----------------------------------------
Product: AVOSKIN Miraculous Refining Serum Anniversary Edition 
Br